In [2]:
# ============================================================
#              SIMPLE SEO ORPHAN PAGE CRAWLER
# ============================================================
#
# No external libraries
# Only Python built-in modules
#
# Features:
# - Reads XML sitemap
# - Supports sitemap index
# - Crawls HTML pages
# - Finds internal links
# - Counts incoming links
# - Finds possible orphan pages
# - Saves report in TXT file
# ============================================================


from urllib.request import Request, urlopen
from urllib.parse import urljoin, urlsplit
from urllib.error import HTTPError, URLError


# ============================================================
# SETTINGS
# ============================================================

SITEMAP_URL = input(
    "Enter Sitemap URL: "
)

MAX_PAGES_TO_CRAWL = 200

REPORT_FILE = "crawler_report.txt"


# ============================================================
# VARIABLES
# ============================================================

sitemap_urls = []

visited_urls = []

pages_to_crawl = []

incoming_links = {}

crawl_errors = []


# ============================================================
# NORMALIZE URL
# ============================================================

def normalize_url(url):

    url = url.strip()

    # Remove spaces
    url = url.replace(" ", "")

    # Remove fragment
    if "#" in url:

        url = url.split("#")[0]

    # Remove trailing slash
    if url.endswith("/"):

        parsed = urlsplit(url)

        if parsed.path != "/":

            url = url.rstrip("/")

    return url



# ============================================================
# DOWNLOAD URL
# ============================================================

def download_url(url):

    try:

        request = Request(
            url,
            headers={
                "User-Agent":
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 "
                "Chrome/120 Safari/537.36",
                "Accept":
                "text/html,application/xhtml+xml,"
                "application/xml;q=0.9,*/*;q=0.8"
            }
        )

        response = urlopen(
            request,
            timeout=20
        )

        data = response.read()

        # Try UTF-8 first
        try:

            content = data.decode(
                "utf-8"
            )

        except:

            content = data.decode(
                "latin-1",
                errors="ignore"
            )

        return content


    except HTTPError as error:

        message = (
            url
            + " | HTTP Error "
            + str(error.code)
        )

        crawl_errors.append(message)

        return ""


    except URLError as error:

        message = (
            url
            + " | URL Error "
            + str(error.reason)
        )

        crawl_errors.append(message)

        return ""


    except Exception as error:

        message = (
            url
            + " | Error "
            + str(error)
        )

        crawl_errors.append(message)

        return ""



# ============================================================
# CHECK INTERNAL URL
# ============================================================

def is_internal_url(
    url,
    website_domain
):

    try:

        parsed = urlsplit(url)

        if parsed.scheme not in [
            "http",
            "https"
        ]:

            return False

        if parsed.hostname == website_domain:

            return True

        return False


    except:

        return False



# ============================================================
# READ SITEMAP
# ============================================================

def read_sitemap(
    sitemap_url
):

    print(
        "\nReading sitemap:"
    )

    print(
        sitemap_url
    )

    content = download_url(
        sitemap_url
    )

    if content == "":

        print(
            "Could not download sitemap."
        )

        return False


    # --------------------------------------------------------
    # CHECK FOR SITEMAP INDEX
    # --------------------------------------------------------

    if "<sitemapindex" in content.lower():

        print(
            "Sitemap Index detected."
        )

        start = 0

        sitemap_files = []


        while True:

            start_pos = content.find(
                "<loc>",
                start
            )

            if start_pos == -1:

                break


            start_pos = start_pos + 5


            end_pos = content.find(
                "</loc>",
                start_pos
            )

            if end_pos == -1:

                break


            child_sitemap = content[
                start_pos:end_pos
            ].strip()


            child_sitemap = normalize_url(
                child_sitemap
            )


            if child_sitemap not in sitemap_files:

                sitemap_files.append(
                    child_sitemap
                )


            start = end_pos + 6


        print(
            "Child sitemaps found:",
            len(sitemap_files)
        )


        # Read every child sitemap

        for child in sitemap_files:

            print(
                "\nReading child sitemap:"
            )

            print(
                child
            )


            child_content = download_url(
                child
            )


            if child_content == "":

                continue


            start = 0


            while True:

                start_pos = child_content.find(
                    "<loc>",
                    start
                )

                if start_pos == -1:

                    break


                start_pos = start_pos + 5


                end_pos = child_content.find(
                    "</loc>",
                    start_pos
                )

                if end_pos == -1:

                    break


                url = child_content[
                    start_pos:end_pos
                ].strip()


                url = normalize_url(
                    url
                )


                if url not in sitemap_urls:

                    sitemap_urls.append(
                        url
                    )


                start = end_pos + 6


    else:

        # ----------------------------------------------------
        # NORMAL SITEMAP
        # ----------------------------------------------------

        print(
            "Normal XML sitemap detected."
        )


        start = 0


        while True:

            start_pos = content.find(
                "<loc>",
                start
            )

            if start_pos == -1:

                break


            start_pos = start_pos + 5


            end_pos = content.find(
                "</loc>",
                start_pos
            )

            if end_pos == -1:

                break


            url = content[
                start_pos:end_pos
            ].strip()


            url = normalize_url(
                url
            )


            if url not in sitemap_urls:

                sitemap_urls.append(
                    url
                )


            start = end_pos + 6


    print(
        "\nSitemap URLs found:",
        len(sitemap_urls)
    )


    if len(sitemap_urls) == 0:

        print(
            "No URLs found in sitemap."
        )

        return False


    return True



# ============================================================
# EXTRACT INTERNAL LINKS
# ============================================================

def extract_links(
    html,
    current_url,
    website_domain
):

    links = []

    html_lower = html.lower()

    position = 0


    while True:

        href_position = html_lower.find(
            "href",
            position
        )


        if href_position == -1:

            break


        equal_position = html.find(
            "=",
            href_position
        )


        if equal_position == -1:

            break


        quote_position = (
            equal_position + 1
        )


        # Skip spaces

        while (
            quote_position < len(html)
            and html[quote_position] == " "
        ):

            quote_position += 1


        if quote_position >= len(html):

            break


        quote = html[
            quote_position
        ]


        # Quoted URL

        if quote == '"' or quote == "'":

            start_url = (
                quote_position + 1
            )


            end_url = html.find(
                quote,
                start_url
            )


            if end_url == -1:

                break


            href = html[
                start_url:end_url
            ]


        else:

            # Unquoted URL

            start_url = quote_position


            end_url = html.find(
                " ",
                start_url
            )


            if end_url == -1:

                end_url = len(html)


            href = html[
                start_url:end_url
            ]


        href = href.strip()


        # ----------------------------------------------------
        # Ignore unwanted links
        # ----------------------------------------------------

        if href != "":

            if not href.startswith("#"):

                if not href.startswith(
                    "mailto:"
                ):

                    if not href.startswith(
                        "tel:"
                    ):

                        if not href.startswith(
                            "javascript:"
                        ):

                            full_url = urljoin(
                                current_url,
                                href
                            )


                            full_url = normalize_url(
                                full_url
                            )


                            if is_internal_url(
                                full_url,
                                website_domain
                            ):

                                if full_url not in links:

                                    links.append(
                                        full_url
                                    )


        position = (
            end_url + 1
        )


    return links



# ============================================================
# CRAWL WEBSITE
# ============================================================

def crawl_website():

    if len(sitemap_urls) == 0:

        return


    first_url = sitemap_urls[0]


    website_domain = urlsplit(
        first_url
    ).hostname


    print(
        "\nWebsite Domain:",
        website_domain
    )


    print(
        "Maximum crawl pages:",
        MAX_PAGES_TO_CRAWL
    )


    # Add sitemap pages to crawl queue

    for url in sitemap_urls:

        if len(pages_to_crawl) >= (
            MAX_PAGES_TO_CRAWL
        ):

            break


        if url not in pages_to_crawl:

            pages_to_crawl.append(
                url
            )


    index = 0


    while index < len(
        pages_to_crawl
    ):


        if len(visited_urls) >= (
            MAX_PAGES_TO_CRAWL
        ):

            break


        current_url = (
            pages_to_crawl[index]
        )


        index += 1


        if current_url in visited_urls:

            continue


        print(
            "Crawling",
            len(visited_urls) + 1,
            "/",
            MAX_PAGES_TO_CRAWL,
            ":",
            current_url
        )


        html = download_url(
            current_url
        )


        visited_urls.append(
            current_url
        )


        if html == "":

            continue


        links = extract_links(
            html,
            current_url,
            website_domain
        )


        for link in links:


            # ------------------------------------------------
            # Count incoming links
            # ------------------------------------------------

            if link not in incoming_links:

                incoming_links[link] = 0


            incoming_links[link] += 1


            # ------------------------------------------------
            # Add new URL to queue
            # ------------------------------------------------

            if link not in pages_to_crawl:

                if len(
                    pages_to_crawl
                ) < MAX_PAGES_TO_CRAWL:

                    pages_to_crawl.append(
                        link
                    )


    print(
        "\nCrawling completed."
    )



# ============================================================
# FIND ORPHAN PAGES
# ============================================================

def find_orphan_pages():

    orphan_pages = []


    for sitemap_url in sitemap_urls:


        # Only check pages that were actually crawled

        if sitemap_url not in visited_urls:

            continue


        # No internal page linked to this page

        if sitemap_url not in incoming_links:

            orphan_pages.append(
                sitemap_url
            )


    return orphan_pages



# ============================================================
# CREATE TXT REPORT
# ============================================================

def create_report(
    orphan_pages
):

    file = open(
        REPORT_FILE,
        "w",
        encoding="utf-8"
    )


    file.write(
        "==================================================\n"
    )

    file.write(
        "             SEO ORPHAN PAGE REPORT\n"
    )

    file.write(
        "==================================================\n\n"
    )


    file.write(
        "Sitemap:\n"
    )

    file.write(
        SITEMAP_URL + "\n\n"
    )


    file.write(
        "Total Sitemap URLs: "
        + str(len(sitemap_urls))
        + "\n"
    )


    file.write(
        "Total Crawled Pages: "
        + str(len(visited_urls))
        + "\n"
    )


    file.write(
        "Total Pages With Internal Links: "
        + str(len(incoming_links))
        + "\n"
    )


    file.write(
        "Possible Orphan Pages: "
        + str(len(orphan_pages))
        + "\n\n"
    )


    file.write(
        "--------------------------------------------------\n"
    )

    file.write(
        "POSSIBLE ORPHAN PAGES\n"
    )

    file.write(
        "--------------------------------------------------\n\n"
    )


    if len(orphan_pages) == 0:

        file.write(
            "No possible orphan pages found.\n"
        )

    else:

        for page in orphan_pages:

            file.write(
                page + "\n"
            )


    file.write(
        "\n--------------------------------------------------\n"
    )

    file.write(
        "CRAWL ERRORS\n"
    )

    file.write(
        "--------------------------------------------------\n\n"
    )


    if len(crawl_errors) == 0:

        file.write(
            "No crawl errors.\n"
        )

    else:

        for error in crawl_errors:

            file.write(
                error + "\n"
            )


    file.write(
        "\n==================================================\n"
    )

    file.write(
        "END OF REPORT\n"
    )

    file.write(
        "==================================================\n"
    )


    file.close()


    print(
        "\nReport saved:"
    )

    print(
        REPORT_FILE
    )



# ============================================================
# DISPLAY RESULTS
# ============================================================

def display_results(
    orphan_pages
):

    print("\n")

    print(
        "=================================================="
    )

    print(
        "             SEO CRAWLER REPORT"
    )

    print(
        "=================================================="
    )


    print(
        "Total Sitemap URLs:",
        len(sitemap_urls)
    )


    print(
        "Total Crawled Pages:",
        len(visited_urls)
    )


    print(
        "Total Pages With Internal Links:",
        len(incoming_links)
    )


    print(
        "Possible Orphan Pages:",
        len(orphan_pages)
    )


    print(
        "=================================================="
    )


    if len(orphan_pages) > 0:

        print(
            "\n---------- POSSIBLE ORPHAN PAGES ----------\n"
        )


        for page in orphan_pages:

            print(page)


    else:

        print(
            "\nNo possible orphan pages found."
        )


    if len(crawl_errors) > 0:

        print(
            "\n---------- CRAWL ERRORS ----------\n"
        )


        for error in crawl_errors:

            print(error)


    print(
        "\n=================================================="
    )



# ============================================================
# MAIN PROGRAM
# ============================================================

print(
    "\nStarting SEO Orphan Page Crawler..."
)

print(
    "------------------------------------------"
)


success = read_sitemap(
    SITEMAP_URL
)


if success:

    crawl_website()

    orphan_pages = find_orphan_pages()

    display_results(
        orphan_pages
    )

    create_report(
        orphan_pages
    )

else:

    print(
        "\nCrawler stopped because sitemap could not be read."
    )


print(
    "\nProgram Finished."
)



Starting SEO Orphan Page Crawler...
------------------------------------------

Reading sitemap:
https://cplusoft.com/sitemap.xml
Normal XML sitemap detected.

Sitemap URLs found: 330

Website Domain: cplusoft.com
Maximum crawl pages: 200
Crawling 1 / 200 : https://cplusoft.com
Crawling 2 / 200 : https://cplusoft.com/about
Crawling 3 / 200 : https://cplusoft.com/ai-lab
Crawling 4 / 200 : https://cplusoft.com/blog
Crawling 5 / 200 : https://cplusoft.com/blog/ai-intelligent-systems/future-of-adaptive-interaction
Crawling 6 / 200 : https://cplusoft.com/blog/ai-intelligent-systems/how-ai-is-reshaping-spoken-interaction
Crawling 7 / 200 : https://cplusoft.com/blog/ai-intelligent-systems/the-governance-gap-in-the-ai-era
Crawling 8 / 200 : https://cplusoft.com/blog/ai-intelligent-systems/the-rise-of-autonomous-coordination
Crawling 9 / 200 : https://cplusoft.com/blog/ai-intelligent-systems/when-ai-ambitions-outspace-organizational-readiness
Crawling 10 / 200 : https://cplusoft.com/blog/data-